# Generating ONNX Training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [ ]:
from pathlib import Path

import onnx
import onnxruntime.training.onnxblock as onnxblock

In [ ]:
# Creating a class with a Loss function
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [ ]:
artifacts_path = Path('./artifacts/resnet18_frozen_layer_3/hailo/')
onnx_artifacts_path = artifacts_path / 'resnet18_quantized_model_hailo.onnx'
onnx_model = onnx.load_model(onnx_artifacts_path)

# for param in onnx_model.graph.initializer:
#     print(param.name)

In [ ]:
training_block = CustomTrainingBlock()
for param in onnx_model.graph.initializer:
    if 'frozen_features' in param.name or 'bn' in param.name or 'downsample.1' in param.name:
        print(param.name.ljust(40), '\t--->\tfrozen')
        training_block.requires_grad(param.name, False)
    else:
        print(param.name.ljust(40), '\t--->\tlearnable')
        training_block.requires_grad(param.name, True)

# Building training graph and eval graph
model_params = None
with onnxblock.base(onnx_model):
    build_output = training_block(*[output.name for output in onnx_model.graph.output])
    print('Graph output nodes:', build_output)
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

In [ ]:
# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / 'checkpoint')

ir_version = 9
opset_import_version = 17

training_model.ir_version = ir_version
# print('Simplifying ONNX training model...')
# training_model, check = simplify(training_model)
# assert check, "Simplified ONNX training model could not be validated"artifacts_optim_2_comp_1
onnx.save(training_model, artifacts_path / 'training_model.onnx')

optimizer_model.ir_version = ir_version
optimizer_model.opset_import[1].version = opset_import_version
onnx.save(optimizer_model, artifacts_path / 'optimizer_model.onnx')

eval_model.ir_version = ir_version
# print('Simplifying ONNX eval model...')
# eval_model, check = simplify(eval_model)
# assert check, "Simplified ONNX eval model could not be validated"
onnx.save(eval_model, artifacts_path / 'eval_model.onnx')

print(f'Artifacts saved in {artifacts_path} directory')

## Verify model

In [ ]:
# from pathlib import Path

# import numpy as np
# from onnxruntime.training.api import CheckpointState, Module, Optimizer

In [ ]:
# # artifacts path
# device = 'cpu'
# artifacts_path = Path(f'../artifacts/{model_name}/artifacts_cpu/frozen_layer_{frozen_layer_num}_classes_{num_classes}')

# print('Loading artifacts...')
# # Create checkpoint state
# state = CheckpointState.load_checkpoint(artifacts_path / 'checkpoint')

# # Create module
# print(f'Creating model with {device} device...')
# model = Module(
#     artifacts_path / 'training_model.onnx',
#     state,
#     artifacts_path / 'eval_model.onnx',
#     device=device,
# )

# # Create optimizer
# optimizer = Optimizer(artifacts_path / 'optimizer_model.onnx', model)
# optimizer.set_learning_rate(0.001)

In [ ]:
# print(f'Model inputs: {model.input_names()}')
# print(f'Model outputs: {model.output_names()}')

# batch_size = 32
# forward_inputs = [
#     np.ones((batch_size, 3, 128, 128), dtype=np.float32),
# ]
# model.train()
# train_loss, logits = model(*forward_inputs)